In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Parameter yang kerap digunakan untuk transpilasi

<details>
<summary><b>Versi pakej</b></summary>

Kod pada halaman ini dibangunkan menggunakan keperluan berikut.
Kami mengesyorkan penggunaan versi ini atau yang lebih baharu.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
Halaman ini menerangkan beberapa parameter yang lebih kerap digunakan untuk transpilasi tempatan. Parameter ini dikonfigurasi menggunakan argumen kepada [`generate_preset_pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager) atau [`transpile`](https://docs.quantum.ibm.com/api/qiskit/compiler#qiskit.compiler.transpile).

<span id="approx-degree"></span>
## Darjah penghampiran
Anda boleh menggunakan darjah penghampiran untuk menentukan sejauh mana anda ingin Circuit hasil sesuai dengan Circuit yang diingini (input). Ini ialah nilai float dalam julat (0.0 - 1.0), di mana 0.0 adalah penghampiran maksimum dan 1.0 (lalai) adalah tanpa penghampiran. Nilai yang lebih kecil menukar ketepatan output dengan kemudahan pelaksanaan (iaitu, lebih sedikit get). Nilai lalai ialah 1.0.

Dalam sintesis dua qubit uniter (digunakan dalam peringkat awal semua tahap dan untuk peringkat pengoptimuman dengan tahap pengoptimuman 3), nilai ini menentukan kesetiaan sasaran output penguraian. Iaitu, berapa banyak ralat yang diperkenalkan apabila perwakilan matriks Circuit ditukar kepada get diskret. Jika darjah penghampiran adalah nilai yang lebih rendah (lebih banyak penghampiran), Circuit output dari sintesis akan lebih berbeza daripada matriks input, tetapi juga berkemungkinan mempunyai lebih sedikit get (kerana sebarang operasi dua qubit sewenang-wenangnya boleh diuraikan dengan sempurna dengan paling banyak tiga get CX) dan lebih mudah untuk dijalankan.

Apabila darjah penghampiran kurang daripada 1.0, Circuit dengan satu atau dua get CX mungkin disintesis, yang membawa kepada lebih sedikit ralat daripada perkakasan, tetapi lebih banyak daripada penghampiran. Memandangkan CX adalah get yang paling mahal dari segi ralat, mungkin bermanfaat untuk mengurangkan bilangannya dengan mengorbankan kesetiaan dalam sintesis (teknik ini digunakan untuk meningkatkan isipadu kuantum pada peranti IBM&reg;: [Validating quantum computers using randomized model circuits](https://arxiv.org/abs/1811.12926)).

Sebagai contoh, kami menjana `UnitaryGate` dua qubit rawak yang akan disintesis pada peringkat awal. Menetapkan `approximation_degree` kurang daripada 1.0 mungkin menjana Circuit penghampiran. Kami juga mesti menentukan `basis_gates` untuk memberitahu kaedah sintesis gate mana yang boleh digunakannya untuk sintesis penghampiran.

In [1]:
from qiskit import QuantumCircuit, QuantumRegister
from qiskit.circuit.library import UnitaryGate
from qiskit.quantum_info import random_unitary
from qiskit.transpiler import generate_preset_pass_manager

UU = random_unitary(4, seed=12345)
rand_U = UnitaryGate(UU)

qubits = QuantumRegister(2, name="q")
qc = QuantumCircuit(qubits)
qc.append(rand_U, qubits)
pass_manager = generate_preset_pass_manager(
    optimization_level=1,
    approximation_degree=0.85,
    basis_gates=["sx", "rz", "cx"],
)
approx_qc = pass_manager.run(qc)
print(approx_qc.count_ops()["cx"])

2


This yields an output of `2` because the approximation requires fewer CX gates.

<span id="seed"></span>
## Random number generator seed

Some parts of the transpiler are stochastic, so repeated transpilation runs may return different results. To obtain a reproducible result, you can set the seed for the pseudorandom number generator using the `seed_transpiler` argument. Repeated runs using the same seed will return the same results.

Example:

In [2]:
pass_manager = generate_preset_pass_manager(
    optimization_level=1, seed_transpiler=11, basis_gates=["sx", "rz", "cx"]
)
optimized_1 = pass_manager.run(qc)
optimized_1.draw("mpl")

<Image src="../docs/images/guides/common-parameters/extracted-outputs/dbc652e8-53a4-47a9-a66e-d9c1e5ef07c9-0.svg" alt="Output of the previous code cell" />

Ini menghasilkan output `2` kerana penghampiran memerlukan lebih sedikit get CX.

<span id="seed"></span>
## Benih penjana nombor rawak
Beberapa bahagian Transpiler adalah stokastik, jadi jalankan transpilasi berulang mungkin mengembalikan keputusan yang berbeza. Untuk mendapatkan keputusan yang boleh diulang, anda boleh menetapkan benih untuk penjana nombor pseudorawak menggunakan argumen `seed_transpiler`. Jalankan berulang menggunakan benih yang sama akan mengembalikan keputusan yang sama.

Contoh:

In [3]:
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke
from qiskit.transpiler import Layout

backend = FakeSherbrooke()

a, b = qubits
initial_layout = Layout({a: 5, b: 6})

pass_manager = generate_preset_pass_manager(
    optimization_level=1, backend=backend, initial_layout=initial_layout
)
transpiled_circ = pass_manager.run(qc)

transpiled_circ.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/common-parameters/extracted-outputs/e18c034c-eb26-4d9d-81d7-37e0eafa17c7-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/common-parameters/extracted-outputs/dbc652e8-53a4-47a9-a66e-d9c1e5ef07c9-0.svg)

<span id="init-layout"></span>
## Susun atur awal
Sebelum transpilasi, qubit yang terkandung dalam Circuit anda adalah qubit maya yang tidak semestinya bersamaan dengan qubit fizikal pada Backend sasaran. Anda boleh menentukan pemetaan awal qubit maya kepada qubit fizikal menggunakan argumen `initial_layout`. Perhatikan bahawa susun atur qubit akhir mungkin berbeza daripada susun atur awal kerana Transpiler mungkin mengatur semula qubit menggunakan get swap atau cara lain.

Dalam contoh di bawah, kami membina susun atur awal untuk backend mock [`FakeSherbrooke`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/fake-provider-fake-sherbrooke#fakesherbrooke) dengan mencipta objek [`Layout`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.Layout). Susun atur kami memetakan qubit pertama Circuit kami kepada qubit 5 Sherbrooke, dan memetakan qubit kedua Circuit kami kepada qubit 6 Sherbrooke. Perhatikan bahawa qubit fizikal sentiasa diwakili oleh integer.

In [4]:
initial_layout = [5, 6]

pass_manager = generate_preset_pass_manager(
    optimization_level=1, backend=backend, initial_layout=initial_layout
)
transpiled_circ = pass_manager.run(qc)

transpiled_circ.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/common-parameters/extracted-outputs/a7800d8a-7354-48e4-a55f-f902ae28c875-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/common-parameters/extracted-outputs/e18c034c-eb26-4d9d-81d7-37e0eafa17c7-0.svg)

Selain menentukan objek Layout, anda juga boleh menghantar senarai integer, di mana elemen ke-$i$ senarai mengandungi qubit fizikal yang qubit ke-$i$ hendak dipetakan. Contohnya:

In [5]:
from qiskit.visualization import plot_error_map

plot_error_map(backend, figsize=(30, 24))

<Image src="../docs/images/guides/common-parameters/extracted-outputs/8df57c6a-1ff4-4d58-9b7e-4378452c3025-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/common-parameters/extracted-outputs/a7800d8a-7354-48e4-a55f-f902ae28c875-0.svg)

Anda boleh menggunakan fungsi [`plot_error_map`](https://docs.quantum.ibm.com/api/qiskit/qiskit.visualization.plot_error_map) untuk menjana gambar rajah graf peranti dengan maklumat ralat dan qubit fizikal yang berlabel. Anda juga boleh melihat gambar rajah yang serupa pada halaman [Sumber Pengiraan](https://quantum.cloud.ibm.com/computers).